[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# What an API Is &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `archive` and
`exchange`. Run it first.


In [1]:
import json
import socket
import urllib.request
from pathlib import Path
from urllib.parse import parse_qs, urlsplit

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if not Path("practice_api.py").exists():      # true in Colab, which starts with only the notebook
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")

import practice_api

BASE = practice_api.start()
host, port = urlsplit(BASE).hostname, urlsplit(BASE).port

archive = ("https://archive-api.open-meteo.com/v1/archive"
           "?latitude=69.65&longitude=18.96&start_date=2025-01-15&end_date=2025-01-17"
           "&daily=temperature_2m_mean&models=era5")


def exchange(request):
    """Send a request as raw text, and return the raw text of the response."""
    with socket.create_connection((host, port)) as conn:
        conn.sendall(request.encode("utf-8"))
        return conn.makefile(encoding="utf-8").read()


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A path, and a query parameter by name.


In [2]:
parts = urlsplit(archive)

print("path:    ", parts.path)
print("end_date:", parse_qs(parts.query)["end_date"][0])


path:     /v1/archive
end_date: 2025-01-17


`parse_qs` returns a list of values for every name, so `[0]` takes the first. `urlsplit` alone
gives the whole query as a single string.


**2.** A method the endpoint does not allow.


In [3]:
lines = ["POST /stations HTTP/1.1", f"Host: {host}:{port}", "Connection: close"]
head, body = exchange("\r\n".join(lines) + "\r\n\r\n").split("\n\n", 1)

status_line, *header_lines = head.splitlines()
headers = dict(line.split(": ", 1) for line in header_lines)

print(status_line)
print("Allow:", headers["Allow"])


HTTP/1.1 405 Method Not Allowed
Allow: GET


Only the method in the request line changed from the notebook's requests. The server read it, found
that `/stations` accepts only `GET`, and said so twice: in the status code, and in the `Allow`
response header.


**3.** Counting the stations.


In [4]:
with urllib.request.urlopen(f"{BASE}/stations") as response:
    stations = json.loads(response.read())

print(len(stations), "stations:", [s["name"] for s in stations])


4 stations: ['Bergen', 'Oslo', 'Svalbard', 'Tromso']


The collection is a JSON list, so it arrives as a Python list, and `len` counts it.


**4.** A station on one line.


In [5]:
with urllib.request.urlopen(f"{BASE}/stations/bergen") as response:
    bergen = json.loads(response.read())

print(bergen["name"], bergen["latitude"], bergen["longitude"])


Bergen 60.39 5.32


Keep `bergen`: task 6 needs its coordinates.


**5.** A 404, read by hand.


In [6]:
lines = ["GET /stations/narvik HTTP/1.1", f"Host: {host}:{port}", "Connection: close"]
head, body = exchange("\r\n".join(lines) + "\r\n\r\n").split("\n\n", 1)

print(head.splitlines()[0])
print(json.loads(body)["error"])


HTTP/1.1 404 Not Found
no station with id 'narvik'


Nothing raised, because `exchange` does not interpret the status code at all: it sends the request
and returns the response as text. A `404` is a response whose status line says so. Deciding that it
counts as a failure is the client's job, which `urlopen` does by raising, and which `requests`
leaves to you.


**6.** Bergen's temperatures from Open-Meteo.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.


In [7]:
url = ("https://archive-api.open-meteo.com/v1/archive"
       f"?latitude={bergen['latitude']}&longitude={bergen['longitude']}"
       "&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5")

with urllib.request.urlopen(url, timeout=30) as response:
    daily = json.loads(response.read())["daily"]

for day, temperature in zip(daily["time"], daily["temperature_2m_mean"]):
    print(day, temperature)


2025-01-15 7.8
2025-01-16 8.0
2025-01-17 8.0


Only the latitude and longitude query parameters changed from the notebook's URL, and the f-string
takes them from the dictionary task 4 fetched. The practice API supplied the place and Open-Meteo
supplied the weather, the same division of labor as the notebook's closing example.


---

&#8592; **Back to:** [What an API Is](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/01-what-an-api-is.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
